# Lectures 13 & 14 — From a GeoPackage to a Finished Map

**Course:** Urban Informatics and Spatial Analytics  
**Instructor:** Varun Varghese  
**Department of Humanities and Social Sciences, IIT Tirupati**

---

## Why are we doing this now?

So far, we have mainly worked with **tabular data** using pandas.

For Test 1, however, you are collecting **spatial point data** using QField. The final output must communicate where the observations are located and what was observed at those locations.

These lectures therefore introduce the minimum spatial-visualisation workflow needed to move from:

```text
FIELD DATA / GEOPACKAGE
        ↓
GEODATAFRAME
        ↓
CHECK GEOMETRY + CRS
        ↓
PLOT THE POINTS
        ↓
ADD CONTEXT + SYMBOLOGY
        ↓
TITLE + LEGEND + NORTH ARROW + SCALE BAR
        ↓
EXPORT A FINISHED MAP
```

We will demonstrate the workflow using a **real meat-shop point dataset from Tirupati city** and then adapt the same logic to the waste-bin data collected for Test 1.

## Learning objectives

By the end of Lectures 13–14, you should be able to:

1. explain the difference between a pandas `DataFrame` and a GeoPandas `GeoDataFrame`
2. read a GeoPackage (`.gpkg`) using GeoPandas
3. inspect its attributes, geometry type, CRS, and spatial extent
4. make a basic point map using `GeoDataFrame.plot()`
5. understand the role of Matplotlib's **figure** and **axes**
6. reproject spatial data for mapping and distance-based map elements
7. add a web basemap using Contextily
8. control marker size, transparency, labels, title, and legend
9. add a simple north arrow and scale bar
10. export a map as a high-resolution PNG or PDF
11. apply exactly the same workflow to your QField waste-bin GeoPackage

# 1. From pandas to GeoPandas

A pandas DataFrame stores rows and columns:

```text
name | category | value
```

A **GeoDataFrame** stores rows and columns **plus a geometry column**:

```text
name | category | value | geometry
```

The geometry may contain:

- **Point** → shop, waste bin, bus stop, accident location
- **LineString** → road, route, river
- **Polygon** → ward, parcel, lake, building footprint

For our meat-shop dataset, each row represents a **meat-shop location**, so the geometry should be a **Point**.

## 1.1 Libraries used in this lecture

We use three main libraries:

```python
geopandas   → read and manipulate spatial vector data
matplotlib  → create and control the figure/map
contextily  → add web basemap tiles
```

These packages are assumed to be available in the course Conda environment.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx

# 2. Read the Tirupati meat-shop GeoPackage

For this lecture, keep `meatshops.gpkg` in the **same folder as this notebook**.

Then we only need the filename:

```text
meatshops.gpkg
```

GeoPandas will read both the attribute data and the spatial information stored in the GeoPackage.

In [ ]:
data_path = "meatshops.gpkg"

meatshops = gpd.read_file(data_path)

`gpd.read_file()` is the spatial-data equivalent of something you already know:

```python
pd.read_csv(...)
```

The difference is that GeoPandas also reads the **geometry** and the **coordinate reference system (CRS)**.

In [ ]:
type(meatshops)

The expected object is:

```text
geopandas.geodataframe.GeoDataFrame
```

So a GeoDataFrame is still very similar to a DataFrame — most pandas operations you already know continue to work.

## 2.1 A GeoPackage can contain one or more layers

A `.gpkg` file is a **container**. It can store:

- one or more spatial layers
- their attribute tables
- geometry
- coordinate reference system information

If you want to inspect the available layers:

In [ ]:
gpd.list_layers(data_path)

If there is only one relevant layer, `gpd.read_file(data_path)` is enough.

If there are several layers, you can specify one:

```python
meatshops = gpd.read_file(data_path, layer="layer_name")
```

In [ ]:
meatshops = gpd.read_file(data_path, layer="notes")

# 3. Inspect the spatial dataset before mapping

Just as with pandas, **do not begin plotting before understanding the data**.

Start with the familiar checks:

In [ ]:
meatshops.head()

In [ ]:
meatshops.shape

In [ ]:
meatshops.columns

In [ ]:
meatshops.info()

Now add the spatial checks.

## 3.1 What type of geometry do we have?

In [ ]:
meatshops.geometry.geom_type.value_counts()

For this dataset, we expect the features to be **points**.

This is the same geometry type your waste-bin data from QField should use: **one point for each bin or bin station**.

## 3.2 What CRS is the dataset using?

The CRS tells us how the coordinate values should be interpreted.

In [ ]:
meatshops.crs

For field data collected using phone GPS, you will commonly encounter a geographic CRS such as **WGS 84 (EPSG:4326)**, where coordinates are stored as longitude and latitude in decimal degrees.

For a finished local map, however, it is often useful to work in a **projected CRS** where units are metres.

## 3.3 What area does the dataset cover?

`total_bounds` returns:

```text
minimum x, minimum y, maximum x, maximum y
```

In [ ]:
meatshops.total_bounds

That gives the bounding rectangle containing all features.

# 4. Your first map: `GeoDataFrame.plot()`

The simplest possible map is:

In [ ]:
meatshops.plot()

This is already a map because the `geometry` column contains spatial coordinates.

But this is only a **quick exploratory plot**. It is not yet a good final map.

A useful final map should provide enough context for someone else to understand:

- **what** the points represent
- **where** they are
- **how** different categories are represented
- the orientation and approximate scale of the map

# 5. Matplotlib: the canvas on which GeoPandas draws

GeoPandas uses **Matplotlib** to display maps.

Before making a polished map, understand two objects:

```text
FIGURE (`fig`) = the entire output image / page

AXES (`ax`)     = the plotting area inside that figure
```

A useful analogy is:

> **Figure = sheet of paper**  
> **Axes = the map frame drawn on that sheet**

We create both with:

```python
fig, ax = plt.subplots(figsize=(9, 8))
```

### What does each part mean?

- `plt` is the short name we gave `matplotlib.pyplot`.
- `subplots()` creates a Figure and one or more Axes.
- `fig` stores the entire figure.
- `ax` stores the particular plotting area we will draw on.
- `figsize=(9, 8)` means approximately **9 inches wide × 8 inches high**.

We need `ax` because several different map elements must be placed on the **same map**:

```text
survey points
      +
basemap
      +
title
      +
legend
      +
north arrow
      +
scale bar
```

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

meatshops.plot(
    ax=ax,
    markersize=45,
    alpha=0.8
)

ax.set_title("Meat-shop locations in Tirupati")

plt.show()

## What is `meatshops.plot()` doing?

`plot()` is a GeoPandas method.

Because `meatshops` is a **GeoDataFrame**, GeoPandas knows that its `geometry` column contains spatial objects.

```python
meatshops.plot(ax=ax)
```

means:

> Take the geometry in `meatshops` and draw it on the Matplotlib Axes called `ax`.

For point data:

- `markersize=45` controls the size of each plotted point.
- `alpha=0.8` controls transparency:
  - `1` = fully opaque
  - `0` = completely transparent
- `ax=ax` is crucial because it tells GeoPandas **which map frame to draw on**.

`plt.show()` asks Matplotlib to display the completed figure.

# 6. Reproject before making a final local map

The original QField / GeoPackage data may be stored using geographic coordinates such as:

```text
longitude, latitude
```

measured in **degrees**.

Degrees are excellent for identifying a global location, but they are inconvenient when we want to add map elements whose size is expressed in **metres**.

For example, a scale bar saying:

```text
1 km
```

is naturally constructed if our map coordinates are measured in metres.

Tirupati lies in **UTM Zone 44N**, so for this local exercise we use:

```text
EPSG:32644
WGS 84 / UTM Zone 44N
Units: metres
```

We create a new GeoDataFrame rather than overwriting the original:

In [ ]:
meatshops_map = meatshops.to_crs(epsg=32644)

meatshops_map.crs

## What does `to_crs()` do?

```python
meatshops.to_crs(epsg=32644)
```

**transforms the coordinates** of every geometry into another Coordinate Reference System.

It does **not** move the shops to different real-world locations.

The same shop is being represented using a different coordinate system.

Conceptually:

```text
Same real location
      ↓
longitude / latitude in degrees
      ↓  to_crs()
Easting / Northing in metres
```

After this transformation:

- `x` behaves like an **Easting**
- `y` behaves like a **Northing**
- distances on this local map can be interpreted in metres
- constructing a simple scale bar becomes much easier

# 7. Add geographic context with a basemap

Our survey points tell us **where the observations are**, but point locations alone do not tell a reader much about the surrounding city.

A **basemap** is a reference map placed underneath our own data.

It can provide context such as:

- roads
- street names
- built-up areas
- neighbourhoods
- landmarks
- terrain or satellite imagery

For this lecture, we will use the standard **ESRI world street map** basemap.

Contextily obtains the map as small web-map image tiles and places them underneath our GeoPandas layer.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

meatshops_map.plot(
    ax=ax,
    markersize=55,
    color="red",
    alpha=0.85,
    edgecolor="black",
    linewidth=0.5
)

ctx.add_basemap(
    ax,
    crs=meatshops_map.crs,
    source=ctx.providers.Esri.WorldStreetMap
)

ax.set_title("Meat-shop locations in Tirupati city")
ax.set_axis_off()

plt.show()

## Understanding `ctx.add_basemap()`

```python
ctx.add_basemap(
    ax,
    crs=meatshops_map.crs,
    source=ctx..providers.Esri.WorldStreetMap
)
```

### `ax`

This tells Contextily:

> Put the basemap on the **same Axes** where the meat-shop points were plotted.

If the points and basemap were drawn on different Axes, they would not appear together.

### `crs=meatshops_map.crs`

The online map tiles and our GeoDataFrame may use different coordinate systems.

By supplying the GeoDataFrame's CRS, Contextily knows how the basemap must be transformed so that:

> a road on the basemap and a meat shop at that road appear in the correct relative locations.

### `source=...`

This chooses the **tile provider / visual style**.

Here:

```python
ctx..providers.Esri.WorldStreetMap
```

uses the familiar standard OpenStreetMap street map.

### Why does this require internet?

The basemap is not stored inside your GeoPackage. Contextily downloads map tiles from an online provider when the cell runs.

Your own points can still be plotted without internet; only the online basemap requires the connection.

## Other basemap options

You can change only the `source=` argument to try another map style.

| Purpose | Contextily provider |
|---|---|
| Standard street map | `ctx.providers.OpenStreetMap.Mapnik` |
| Humanitarian OSM style | `ctx.providers.OpenStreetMap.HOT` |
| Topographic context | `ctx.providers.OpenTopoMap` |
| Esri street map | `ctx.providers.Esri.WorldStreetMap` |
| Satellite / aerial imagery | `ctx.providers.Esri.WorldImagery` |
| Light, subdued background | `ctx.providers.CartoDB.Positron` |
| More detailed CARTO street style | `ctx.providers.CartoDB.Voyager` |

For **Test 1**, Esri street map is a good default because campus roads and surrounding context are easy to recognise.

A basemap should support the information you are trying to communicate. It should not visually overpower your own data.

Provider availability and styling can change over time; these provider names are currently available through Contextily's `xyzservices` provider catalogue.

# 8. Essential map elements: title, legend, north arrow, and scale bar

A plotted dataset and a finished map are not quite the same thing.

A reader who did not collect your data should be able to understand the map without asking you questions.

Four basic elements are especially useful:

### Title

Answers:

> **What is being shown, and where?**

Example:

```text
Mapped Meat Shops in Tirupati City
```

### Legend

Answers:

> **What do the symbols or colours mean?**

In the meat-shop example every point represents a meat shop, so one legend entry is sufficient.

In your waste-bin map, a legend becomes even more important if colours distinguish categories such as:

```text
Segregated
Not segregated
Partial
```

### North arrow

Answers:

> **Which direction is north?**

It provides orientation.

### Scale bar

Answers:

> **What real-world distance does a length on this map represent?**

It helps a reader judge whether two points are tens of metres, hundreds of metres, or kilometres apart.

# 9. Building a north-arrow function

We will create a small function because we may want to place the same type of north arrow on several maps.

```python
def add_north_arrow(ax, x=0.95, y=0.92):
```

### Parameters

- `ax` → the Matplotlib Axes on which the arrow should be drawn.
- `x=0.95` → horizontal position of the arrow.
- `y=0.92` → vertical position of the arrow.

The values `0.95` and `0.92` are **not map coordinates**.

We will use:

```python
xycoords="axes fraction"
```

which treats the map frame as a coordinate system from `0` to `1`:

```text
(0,1) ---------------------- (1,1)
  |                            |
  |          MAP FRAME         |
  |                            |
(0,0) ---------------------- (1,0)
```

Therefore:

```text
x = 0.95
```

means 95% of the way from the left edge,

and:

```text
y = 0.92
```

means 92% of the way from the bottom.

This is useful because the arrow remains near the upper-right corner even if the map CRS or coordinate values change.

In [ ]:
def add_north_arrow(ax, x=0.95, y=0.92):
    """Add a simple north arrow to a Matplotlib Axes."""

    ax.annotate(
        "N",
        xy=(x, y),
        xytext=(x, y - 0.12),
        xycoords="axes fraction",
        ha="center",
        va="center",
        fontsize=16,
        fontweight="bold",
        arrowprops=dict(
            facecolor="black",
            edgecolor="black",
            width=4,
            headwidth=12
        )
    )

## Reading the north-arrow code

### `ax.annotate()`

`annotate()` is a Matplotlib method for placing **text and/or an arrow** on an Axes.

```python
"N"
```

is the text that will be displayed.

### `xy=(x, y)`

This is the location toward which the arrow points.

### `xytext=(x, y - 0.12)`

This places the `"N"` text slightly below the arrow head.

### `xycoords="axes fraction"`

Tells Matplotlib that `x` and `y` are fractions of the map frame rather than projected Easting/Northing values.

### `ha` and `va`

These mean:

```text
ha = horizontal alignment
va = vertical alignment
```

`"center"` centers the text on the specified position.

### `fontsize` and `fontweight`

Control how the `N` is displayed.

### `arrowprops`

This dictionary controls the appearance of the arrow:

- `facecolor` → inside colour
- `edgecolor` → outline colour
- `width` → shaft width
- `headwidth` → width of the arrow head

The important concept is not memorising every argument.

It is understanding that the north arrow is simply another graphical object being added to the **same `ax`**.

# 10. Building a scale-bar function

A scale bar is more closely connected to the map coordinate system.

If the GeoDataFrame is in **EPSG:32644**, its coordinates are measured in metres.

Therefore, a horizontal line that is exactly:

```text
1000 coordinate units long
```

represents approximately:

```text
1000 m = 1 km
```

That is why we reprojected the map before constructing the scale bar.

In [ ]:
def add_scale_bar(ax, length_m=1000):
    """Add a simple scale bar when map coordinates are in metres."""

    # 1. Find the current map extent
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    # 2. Choose a position slightly inside the lower-left corner
    x_start = xmin + 0.07 * (xmax - xmin)
    y_start = ymin + 0.07 * (ymax - ymin)

    # 3. Draw a horizontal line whose length equals length_m
    ax.plot(
        [x_start, x_start + length_m],
        [y_start, y_start],
        color="black",
        linewidth=4
    )

    # 4. Create a readable text label
    if length_m >= 1000:
        label = f"{length_m / 1000:g} km"
    else:
        label = f"{length_m:g} m"

    # 5. Place the label above the middle of the scale bar
    ax.text(
        x_start + length_m / 2,
        y_start + 0.015 * (ymax - ymin),
        label,
        ha="center",
        va="bottom",
        fontsize=10
    )

## Reading the scale-bar code

### `ax.get_xlim()` and `ax.get_ylim()`

These ask Matplotlib:

> What is the current geographic extent visible on the map?

They return:

```text
xmin, xmax
ymin, ymax
```

Because the map is projected in metres, these are Easting/Northing values measured in metres.

### `(xmax - xmin)` and `(ymax - ymin)`

These calculate the **width and height of the visible map extent**.

### Why multiply by `0.07`?

```python
x_start = xmin + 0.07 * (xmax - xmin)
```

means:

> Start the scale bar 7% inside the left edge.

The same logic places it 7% above the bottom edge.

This avoids drawing the bar directly on the border.

### `ax.plot(...)`

We use `plot()` here to draw one horizontal black line.

The start is:

```python
x_start
```

and the end is:

```python
x_start + length_m
```

If `length_m=1000`, the line is exactly 1000 projected metres long.

### The `if` statement

You already learned `if` earlier.

Here it makes the text easier to read:

- `1000` m becomes `"1 km"`
- `200` m stays `"200 m"`

### `ax.text(...)`

Adds the scale label above the centre of the line.

Again, the important idea is:

> the scale bar works correctly only because we know the map units are metres.

## Choosing an appropriate scale-bar length

The length should make sense for the size of the study area.

For the Tirupati city demonstration:

```python
add_scale_bar(ax, length_m=1000)
```

gives a **1 km** bar.

For the much smaller IIT Tirupati campus:

```python
add_scale_bar(ax, length_m=100)
```

or:

```python
add_scale_bar(ax, length_m=200)
```

may be more appropriate.

A 10 km scale bar would be meaningless on a campus map, just as a 10 m scale bar would be unhelpful on a city-wide map.

# 11. Build the finished meat-shop map

We can now combine all the pieces.

Read the code in layers:

```text
1. Create figure + axes
2. Plot our own spatial data
3. Add OSM basemap underneath
4. Add title
5. Add legend
6. Add north arrow
7. Add scale bar
8. Remove unnecessary coordinate axes
9. Display map
```

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# 1. Plot the survey points
meatshops_map.plot(
    ax=ax,
    markersize=65,
    alpha=0.9,
    edgecolor="black",
    linewidth=0.6,
    label="Meat shop"
)

# 2. Add OpenStreetMap underneath the survey layer
ctx.add_basemap(
    ax,
    crs=meatshops_map.crs,
    source=ctx.providers.Esri.WorldStreetMap
)

# 3. Explain what the map shows
ax.set_title(
    "Mapped Meat Shops in Tirupati City",
    fontsize=16,
    fontweight="bold",
    pad=15
)

# 4. Explain the point symbol
ax.legend(loc="lower right")

# 5. Add orientation and distance information
add_north_arrow(ax)
add_scale_bar(ax, length_m=1000)

# 6. Hide raw Easting/Northing axes for a cleaner final map
ax.set_axis_off()

plt.tight_layout()
plt.show()

## Understanding the remaining map commands

### `ax.set_title(...)`

Adds the map title.

- `fontsize=16` controls its size.
- `fontweight="bold"` makes it bold.
- `pad=15` adds some space between the title and the map.

### `ax.legend(loc="lower right")`

Creates a legend using the label supplied when the points were plotted:

```python
label="Meat shop"
```

`loc="lower right"` controls where the legend is placed.

### `ax.set_axis_off()`

Hides the raw coordinate axes and tick labels.

Those Easting/Northing numbers are useful during analysis, but they are often unnecessary on a simple final communication map.

### `plt.tight_layout()`

Asks Matplotlib to adjust spacing so titles and plot elements fit more neatly inside the figure.

# 12. Export the map

Displaying a map inside Jupyter is useful while analysing it.

For submission, we need an actual image file.

Matplotlib uses:

```python
plt.savefig(...)
```

The export command should be executed **before** `plt.show()` in the final cell.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

meatshops_map.plot(
    ax=ax,
    markersize=65,
    alpha=0.9,
    edgecolor="black",
    linewidth=0.6,
    label="Meat shop"
)

ctx.add_basemap(
    ax,
    crs=meatshops_map.crs,
    source=ctx.providers.Esri.WorldStreetMap
)

ax.set_title(
    "Mapped Meat Shops in Tirupati City",
    fontsize=16,
    fontweight="bold"
)

ax.legend(loc="lower right")
add_north_arrow(ax)
add_scale_bar(ax, length_m=1000)
ax.set_axis_off()

plt.tight_layout()

plt.savefig(
    "output/tirupati_meatshops_map.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Understanding `savefig()`

```python
plt.savefig(
    "tirupati_meatshops_map.png",
    dpi=300,
    bbox_inches="tight"
)
```

### Filename

```python
"tirupati_meatshops_map.png"
```

tells Python what to call the output image.

Unless a different folder is specified, it is saved in the current working folder.

### `dpi=300`

`dpi` means **dots per inch**.

Higher DPI produces more pixels for a given physical output size.

`300 dpi` is a useful choice for a map that may be submitted in a report or printed.

### `bbox_inches="tight"`

Tells Matplotlib to trim unnecessary empty space around the exported figure.

You can also save a PDF simply by changing the filename:

```python
plt.savefig("tirupati_meatshops_map.pdf", bbox_inches="tight")
```